# 00 â€” Data Exploration

Understand the MTSamples dataset before building the pipeline.

**What this notebook covers**
- Load and inspect MTSamples
- Specialty and note-type distributions
- Text length statistics
- Sample notes per specialty
- Data quality checks

## 1. Setup

In [2]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.utils.config import Paths, settings
from src.utils.logger import get_logger
from src.etl.extract import load_mtsamples

logger = get_logger('notebook')
print('Setup complete')

Setup complete


## 2. Load MTSamples

Download from Kaggle first if you haven't already:
```
https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions
```
Save as `data/raw/mtsamples.csv`

In [3]:
df = load_mtsamples()
print(f'Shape      : {df.shape}')
print(f'Columns    : {list(df.columns)}')
print(f'Specialties: {df["specialty"].nunique()}')
df.head(3)

2026-06-17 11:43:50 | INFO     | src.etl.extract                | Loading MTSamples from C:\Users\Hp\Documents\clinical-nlp-pipeline\data\raw\mtsamples.csv
2026-06-17 11:43:50 | INFO     | src.etl.extract                | MTSamples loaded: 4999 notes, 40 specialties
Shape      : (4999, 6)
Columns    : ['description', 'specialty', 'sample_name', 'transcription', 'keywords', '_source']
Specialties: 40


,description,specialty,sample_name,transcription,keywords,_source
0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller...",mtsamples
1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh...",mtsamples
2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart...",mtsamples


## 3. Specialty Distribution

In [4]:
specialty_counts = (
    df['specialty']
    .value_counts()
    .rename_axis('specialty')
    .reset_index(name='count')
)

fig = px.bar(
    specialty_counts.head(20),
    x='count', y='specialty',
    orientation='h',
    title='Top 20 medical specialties in MTSamples',
    labels={'count': 'Number of notes', 'specialty': ''},
    height=500,
)
fig.update_layout(yaxis={'autorange': 'reversed'})
fig.show()

## 4. Text Length Distribution

In [5]:
df['word_count'] = df['transcription'].dropna().apply(lambda t: len(t.split()))

fig = px.histogram(
    df, x='word_count',
    nbins=60,
    title='Distribution of note word counts',
    labels={'word_count': 'Words per note'},
)
fig.show()

print(df['word_count'].describe().round(0).astype(int))

count    4966
mean      465
std       316
min         1
25%       241
50%       398
75%       615
max      3029
Name: word_count, dtype: int64


## 5. Missing Data Check

In [6]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

# Notes with empty transcription
empty = df['transcription'].isna() | (df['transcription'].str.strip() == '')
print(f'\nEmpty transcriptions: {empty.sum()} ({empty.mean():.1%})')

Missing values per column:
transcription      33
keywords         1068
word_count         33
dtype: int64

Empty transcriptions: 33 (0.7%)


## 6. Sample Notes

Print one note from each of the three most common specialties.

In [7]:
top3 = df['specialty'].value_counts().head(3).index.tolist()

for spec in top3:
    sample = df[df['specialty'] == spec]['transcription'].dropna().iloc[0]
    print(f'\n{"="*60}')
    print(f'  Specialty: {spec}')
    print(f'  Words: {len(sample.split())}')
    print(f'  Preview:')
    print(sample[:400])
    print('...')


  Specialty:  Surgery
  Words: 941
  Preview:
PREOPERATIVE DIAGNOSES:,1.  Hallux rigidus, left foot.,2.  Elevated first metatarsal, left foot.,POSTOPERATIVE DIAGNOSES:,1.  Hallux rigidus, left foot.,2.  Elevated first metatarsal, left foot.,PROCEDURE PERFORMED:,1.  Austin/Youngswick bunionectomy with Biopro implant.,2.  Screw fixation, left foot.,HISTORY: , This 51-year-old male presents to ABCD General Hospital with the above chief complaint
...

  Specialty:  Consult - History and Phy.
  Words: 466
  Preview:
SUBJECTIVE:,  The patient presents with Mom for a first visit to our office for a well-child check with concern of some spitting up quite a bit.  Mom wants to make sure that this is normal.  The patient is nursing well every two to three hours.  She does have some spitting up on occasion.  It has happened two or three times with some curdled appearance x 1.  No projectile in nature, nonbilious.  N
...

  Specialty:  Cardiovascular / Pulmonary
  Words: 68
  Preview:
2-D M-MODE: 